# Tuning global y ensemble final

Tunea por separado `i3_tree_safe_mix` y `v2_physics_by_jet`, rehace el ensemble entre ambos y genera submission.

In [1]:
from __future__ import annotations

from pathlib import Path
import importlib.util
import sys

import pandas as pd
from IPython.display import display


def find_stage_dir(start: Path | None = None) -> Path:
    current = (start or Path.cwd()).resolve()
    for candidate in [current, *current.parents]:
        direct = candidate / "01_tune_global_and_ensemble.py"
        nested = candidate / "15.tune_global" / "01_tune_global_and_ensemble.py"
        if direct.exists():
            return candidate
        if nested.exists():
            return candidate / "15.tune_global"
    raise FileNotFoundError("No se pudo ubicar 15.tune_global/01_tune_global_and_ensemble.py")


stage_dir = find_stage_dir()
project_root = stage_dir.parent
module_path = stage_dir / "01_tune_global_and_ensemble.py"
spec = importlib.util.spec_from_file_location("tune_global_and_ensemble", module_path)
module = importlib.util.module_from_spec(spec)
assert spec.loader is not None
spec.loader.exec_module(module)

pd.set_option("display.max_columns", 200)
pd.set_option("display.width", 240)
print(f"stage_dir={stage_dir}")

stage_dir=/home/santig14/fing/taa/1_TallerAA/15.tune_global


## Ejecutar tuning y ensemble

In [2]:
outputs = module.run_tuning_and_ensemble()

print("Mejores modelos tuneados")
display(outputs["best_models"].round(6))

print("Top 15 blends")
display(outputs["blend_sweep"].head(15).round(6))

i3_tree_safe_mix: best_stage=regularization candidate=reg_lambda=10.0 ams=1.113891 auc=0.909336


v2_physics_by_jet: best_stage=capacity candidate=learning_rate=0.07,n_estimators=330 ams=1.157615 auc=0.908828


Mejores modelos tuneados


,dataset,stage,candidate,params_json,fit_seconds,feature_count,ams_threshold,ams,signal_weight,background_weight,predicted_signal_count,accuracy,precision,recall,f1,auc
0,i3_tree_safe_mix,regularization,reg_lambda=10.0,"{""colsample_bytree"": 0.9, ""eval_metric"": ""logl...",7.371979,81,0.880274,1.113891,22.659409,396.335649,3824,0.78892,0.930178,0.415198,0.574126,0.909336
1,v2_physics_by_jet,capacity,"learning_rate=0.07,n_estimators=330","{""colsample_bytree"": 0.9, ""eval_metric"": ""logl...",3.432264,86,0.897884,1.157615,18.797123,247.474373,3311,0.77520,0.945032,0.365239,0.526856,0.908828


Top 15 blends


,blend_key,blend_type,i3_weight,v2_weight,ams_threshold,ams,signal_weight,background_weight,predicted_signal_count,accuracy,precision,recall,f1,auc
0,raw_probability_i3_0.00_v2_1.00,raw_probability,0.00,1.00,0.897884,1.157615,18.797123,247.474373,3311,0.77520,0.945032,0.365239,0.526856,0.908828
1,percentile_rank_i3_0.00_v2_1.00,percentile_rank,0.00,1.00,0.867600,1.157615,18.797123,247.474373,3311,0.77520,0.945032,0.365239,0.526856,0.908828
2,percentile_rank_i3_0.20_v2_0.80,percentile_rank,0.20,0.80,0.866704,1.156876,18.998800,253.439174,3343,0.77608,0.944062,0.368390,0.529975,0.909277
3,raw_probability_i3_0.19_v2_0.81,raw_probability,0.19,0.81,0.897361,1.156515,18.992789,253.439174,3339,0.77592,0.943995,0.367923,0.529481,0.909257
4,raw_probability_i3_0.23_v2_0.77,raw_probability,0.23,0.77,0.897614,1.156502,18.938383,251.920265,3335,0.77592,0.944528,0.367690,0.529323,0.909327
5,raw_probability_i3_0.22_v2_0.78,raw_probability,0.22,0.78,0.897545,1.156364,18.938383,251.984325,3336,0.77588,0.944245,0.367690,0.529278,0.909308
6,raw_probability_i3_0.01_v2_0.99,raw_probability,0.01,0.99,0.898075,1.156297,18.775483,247.474373,3308,0.77508,0.944982,0.364889,0.526484,0.908854
7,percentile_rank_i3_0.01_v2_0.99,percentile_rank,0.01,0.99,0.867797,1.156297,18.775483,247.474373,3308,0.77508,0.944982,0.364889,0.526484,0.908854
8,raw_probability_i3_0.18_v2_0.82,raw_probability,0.18,0.82,0.897468,1.156195,18.987483,253.439174,3337,0.77584,0.943962,0.367690,0.529234,0.909240
9,raw_probability_i3_0.17_v2_0.83,raw_probability,0.17,0.83,0.897317,1.156195,18.987483,253.439174,3337,0.77584,0.943962,0.367690,0.529234,0.909220


## Comparacion contra mejor ensemble anterior

In [3]:
previous_blend = pd.read_csv(project_root / "13.comb" / "results" / "01_blend_sweep_metrics.csv").iloc[0]
current_blend = outputs["blend_sweep"].iloc[0]

comparison_df = pd.DataFrame([
    {
        "model": "13.comb previous best",
        "blend": str(previous_blend["blend_key"]),
        "auc": float(previous_blend["auc"]),
        "ams": float(previous_blend["ams"]),
        "precision": float(previous_blend["precision"]),
        "recall": float(previous_blend["recall"]),
    },
    {
        "model": "15.tune_global tuned best",
        "blend": str(current_blend["blend_key"]),
        "auc": float(current_blend["auc"]),
        "ams": float(current_blend["ams"]),
        "precision": float(current_blend["precision"]),
        "recall": float(current_blend["recall"]),
    },
])
base = comparison_df.iloc[0]
for metric in ["auc", "ams", "precision", "recall"]:
    comparison_df[f"delta_{metric}_vs_13comb"] = comparison_df[metric] - float(base[metric])

comparison_path = stage_dir / "results" / "02_final_comparison.csv"
comparison_df.to_csv(comparison_path, index=False)
display(comparison_df.round(6))
print(f"Comparacion guardada en: {comparison_path}")

,model,blend,auc,ams,precision,recall,delta_auc_vs_13comb,delta_ams_vs_13comb,delta_precision_vs_13comb,delta_recall_vs_13comb
0,13.comb previous best,percentile_rank_i3_0.30_v2_0.70,0.910106,1.118636,0.943820,0.362787,0.000000,0.000000,0.000000,0.000000
1,15.tune_global tuned best,raw_probability_i3_0.00_v2_1.00,0.908828,1.157615,0.945032,0.365239,-0.001278,0.038979,0.001211,0.002451


Comparacion guardada en: /home/santig14/fing/taa/1_TallerAA/15.tune_global/results/02_final_comparison.csv


## Archivos finales

In [4]:
metadata = outputs["metadata"]
print(f"Submission: {metadata['submission_path']}")
print("Validacion submission:")
display(pd.DataFrame([metadata["submission_validation"]]))

Submission: /home/santig14/fing/taa/1_TallerAA/15.tune_global/results/higgs_submission_tuned_i3_plus_tuned_v2.csv
Validacion submission:


,rows,rank_min,rank_max,signal_count,background_count
0,550000,1,550000,71414,478586
